# صَوْتَك — اختيار صوت عراقي مختلف لـ Habibi

هذا الدفتر يغيّر **صوت المتحدث** باستخدام تسجيل مرجعي عراقي تملكه أو مأذون لك باستخدامه. اللهجة تبقى عراقية لأن النموذج `Specialized/IRQ`.

**المطلوب:** تسجيل نظيف لشخص واحد من 5 إلى 10 ثوانٍ (المقبول 3–12 ثانية)، بدون موسيقى؛ وكتابة **نفس الكلمات التي قالها المتحدث حرفياً** في حقل `reference_text`. بعدها اكتب الكلام الجديد في `script`.

اختار T4 GPU إن كان متاحاً، ثم شغّل خلية 1 وبعدها خلية 2. عند خلية 2 راح يفتح اختيار ملف من الموبايل. الملفات المرفوعة إلى Colab تبقى ببيئة تشغيل جلسة Google؛ لا ترفع تسجيلات خاصة بدون إذن.

ملاحظة: تغيير الصوت لا يضمن تطابقه 100% أو خلو النطق من الأخطاء. Google Colab يفرض حصصاً على الموارد المجانية.


In [ ]:
#@title 1) تثبيت نموذج الصوت العراقي
%pip -q install habibi-tts
%pip -q install 'torch==2.8.0' 'torchvision==0.23.0' 'torchaudio==2.8.0'
!which habibi-tts_infer-cli


In [ ]:
#@title 2) ارفع عينة صوت عراقي واكتب نصها بالضبط
from pathlib import Path
import re, subprocess, tempfile
from IPython.display import Audio, display
from google.colab import files

reference_text = "هلا والله، شلونكم شباب؟ اليوم راح نجرّب التعليق الصوتي باللهجة العراقية." #@param {type:"string"}
script = "هلا والله طلاب، شلونكم؟ اليوم راح نشرح موضوع مهم بالكيمياء، خطوة بخطوة." #@param {type:"string"}

assert reference_text.strip(), "اكتب نفس الكلام الموجود بالتسجيل المرجعي حرفياً."
assert script.strip(), "اكتب السكربت المراد توليده."
assert len(script) <= 1600, "بهذه التجربة الحد 1600 حرف؛ جرّب نصاً أقصر أولاً."
print("اختار ملف تسجيل صوت عراقي بصوت واحد (3–12 ثانية)، من موبايلك. استخدم صوتك أو صوتاً عندك إذن باستخدامه.")
uploaded = files.upload()
assert len(uploaded) == 1, "ارفع ملفاً صوتياً واحداً فقط."
filename, raw_audio = next(iter(uploaded.items()))
ext = Path(filename).suffix.lower()
assert ext in {".wav", ".mp3", ".m4a", ".ogg", ".flac", ".aac"}, "اختر ملفاً صوتياً من الأنواع WAV/MP3/M4A/OGG/FLAC/AAC."
assert len(raw_audio) <= 20*1024*1024, "ملف الصوت كبير؛ استخدم تسجيل قصير."

def chunks_iraqi(text, max_chars=155):
    words = re.sub(r"\s+", " ", text.strip()).split(" ")
    out, current = [], ""
    for word in words:
        if len(word) > max_chars: raise ValueError("كلمة طويلة جداً داخل النص.")
        if current and len(current) + 1 + len(word) > max_chars:
            out.append(current)
            current = ""
        current = f"{current} {word}".strip()
        if len(current) >= 65 and re.search(r"[.!؟؛…:]$", word):
            out.append(current)
            current = ""
    if current: out.append(current)
    return out

with tempfile.TemporaryDirectory() as tmp:
    folder = Path(tmp)
    source_audio = folder / ("voice" + ext)
    source_audio.write_bytes(raw_audio)
    reference_audio = folder / "reference.wav"
    conv = subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(source_audio), "-ar", "24000", "-ac", "1", str(reference_audio)], capture_output=True, text=True)
    if conv.returncode != 0:
        raise RuntimeError("تعذر قراءة التسجيل الصوتي: " + conv.stderr[-1200:])
    probe = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(reference_audio)], capture_output=True, text=True)
    duration = float(probe.stdout.strip())
    assert 3 <= duration <= 12, f"التسجيل طوله {duration:.1f} ثانية. استخدم تسجيل 3–12 ثانية حتى يبقى النص المرجعي مطابقاً للصوت."
    chunks = chunks_iraqi(script)
    gen_file = folder / "text.txt"
    gen_file.write_text("\n".join("[main] " + chunk for chunk in chunks), encoding="utf-8")
    print(f"طول العينة: {duration:.1f} ثانية • مقاطع السكربت: {len(chunks)} • بدأ توليد الصوت العراقي الجديد...")
    command = [
        "habibi-tts_infer-cli", "--model", "Specialized", "--dialect", "IRQ",
        "--ref_audio", str(reference_audio), "--ref_text", reference_text.strip(),
        "--gen_file", str(gen_file), "--output_dir", tmp, "--output_file", "new_iraqi_voice.wav"
    ]
    run = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace", timeout=1200)
    if run.returncode != 0:
        print("آخر أسطر الخطأ من Habibi:")
        print(run.stdout[-12000:])
        raise RuntimeError("فشل التوليد؛ انسخ آخر سطور الخطأ الظاهرة.")
    output = folder / "new_iraqi_voice.wav"
    assert output.exists() and output.stat().st_size > 100, "ما انولد صوت؛ راجع سجل النموذج."
    print("✅ تم توليد الصوت. تگدر تسمعه وتحمّله:")
    display(Audio(filename=str(output)))
    saved = Path("/content/sawtak_custom_iraqi_voice.wav")
    saved.write_bytes(output.read_bytes())
    files.download(str(saved))
